In [0]:
import base64, hashlib, json, zlib

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("run_open_ts", "")
dbutils.widgets.text("silver_update_id", "")
RUN = dbutils.widgets.get("run_id").strip()
RUN_OPEN_TS = dbutils.widgets.get("run_open_ts").strip()
SILVER_UPDATE_ID = dbutils.widgets.get("silver_update_id").strip()
assert RUN.startswith("dq4_silver_") and RUN.replace("_", "").isalnum(), RUN
assert RUN_OPEN_TS and SILVER_UPDATE_ID
LANE = "dq4_exemplars"
PAYLOAD = """..."""
items = json.loads(zlib.decompress(base64.b64decode(PAYLOAD)).decode())
def adapt(value):
    if isinstance(value, str):
        return (value.replace("dq4_silver_20260825", RUN)
                     .replace("2026-08-25T10:50:49.897Z", RUN_OPEN_TS)
                     .replace("f5c7c7ab-e37d-4a31-b9c2-b7631becb16a", SILVER_UPDATE_ID))
    if isinstance(value, list):
        return [adapt(v) for v in value]
    if isinstance(value, dict):
        return {k: adapt(v) for k, v in value.items()}
    return value
items = adapt(items)

def qs(value):
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"

for seq, item in enumerate(items):
    spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log VALUES
      ({qs(RUN)},{qs(LANE)},{qs(item['file'])},{seq},{qs(hashlib.sha256(item['sql'].encode()).hexdigest())},
       'attempted',NULL,NULL,current_timestamp(),NULL,'DQ4')""")
    try:
        spark.sql(item["sql"]).collect()
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log VALUES
          ({qs(RUN)},{qs(LANE)},{qs(item['file'])},{seq},{qs(hashlib.sha256(item['sql'].encode()).hexdigest())},
           'ok',NULL,NULL,current_timestamp(),current_timestamp(),'DQ4')""")
    except Exception as exc:
        msg = str(exc)[:4000]
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log VALUES
          ({qs(RUN)},{qs(LANE)},{qs(item['file'])},{seq},{qs(hashlib.sha256(item['sql'].encode()).hexdigest())},
           'error',{qs(msg)},NULL,current_timestamp(),current_timestamp(),'DQ4')""")
        raise
print({"statements": len(items), "status": "ok"})